# Embedding (SBERT) + Indexing ke ChromaDB

Lanjutan dari `01_preprocessing.ipynb`.

Alur:
1. Load `resep_5000_clean.parquet`
2. Load model `LazarusNLP/all-indo-e5-small-v4` (E5 -> butuh prefix `passage:` / `query:`)
3. Encode `text_for_embedding` -> vektor
4. Buat/isi collection ChromaDB dengan embedding + metadata
5. Tes retrieval sederhana (belum reranking, itu di notebook 03)

In [1]:
# !pip install sentence-transformers chromadb -q

import pandas as pd
import numpy as np
import json
from sentence_transformers import SentenceTransformer
import chromadb

pd.set_option("display.max_colwidth", 150)

c:\Users\Mayceren\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load Dataset Hasil Preprocessing

In [2]:
df = pd.read_parquet("resep_5000_clean.parquet")
print(df.shape)
df[["title_clean", "ingredient_names", "text_for_embedding"]].head(3)

(5000, 12)


,title_clean,ingredient_names,text_for_embedding
0,Martabak Mie,"[mie instan, wortel, buncis, telur ayam, kornet kambing, seledri, daun bawang, kulit lumpia]","martabak mie - bahan: mie instan, wortel, buncis, telur ayam, kornet kambing, seledri, daun bawang, kulit lumpia"
1,Telur Bawang Kecap - Bekal Suami,"[telur, daun jeruk, bawang merah, bawang putih, cabai keriting merah, cabai rawit domba, bawang daun, minyak, kecap manis, saos tiram, saos sambal...","telur bawang kecap - bekal suami - bahan: telur, daun jeruk, bawang merah, bawang putih, cabai keriting merah, cabai rawit domba, bawang daun, min..."
2,166 : Korean Mayak Eggs,"[telur ayam, lombok merah keriting, lombok hijau keriting, daun bawang, bawang putih, wijen sangrai, minyak wijen, kecap asin, madu, air]","166 : korean mayak eggs - bahan: telur ayam, lombok merah keriting, lombok hijau keriting, daun bawang, bawang putih, wijen sangrai, minyak wijen,..."


In [3]:
# beri recipe_id unik & stabil, dipakai sbg id di ChromaDB
df = df.reset_index(drop=True)
df["recipe_id"] = df.index.map(lambda i: f"resep_{i}")
df["recipe_id"].head()

0    resep_0
1    resep_1
2    resep_2
3    resep_3
4    resep_4
Name: recipe_id, dtype: object

## 2. Load Model SBERT (E5)

E5 wajib pakai prefix:
- `"passage: "` untuk teks yang disimpan/di-index (dokumen resep)
- `"query: "` untuk teks pencarian dari user

In [4]:
MODEL_NAME = "LazarusNLP/all-indo-e5-small-v4"
model = SentenceTransformer(MODEL_NAME)
print("Embedding dimension:", model.get_sentence_embedding_dimension())

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3257.25it/s]


Embedding dimension: 384


C:\Users\Mayceren\AppData\Local\Temp\ipykernel_25464\3726833434.py:3: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", model.get_sentence_embedding_dimension())


In [5]:
def add_passage_prefix(text: str) -> str:
    return f"passage: {text}"


def add_query_prefix(text: str) -> str:
    return f"query: {text}"


df["embedding_input"] = df["text_for_embedding"].apply(add_passage_prefix)
df["embedding_input"].iloc[0]

'passage: martabak mie - bahan: mie instan, wortel, buncis, telur ayam, kornet kambing, seledri, daun bawang, kulit lumpia'

## 3. Encode Semua Resep

`normalize_embeddings=True` supaya konsisten dipakai dengan cosine similarity di ChromaDB.

In [6]:
embeddings = model.encode(
    df["embedding_input"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True,
)
print(embeddings.shape)

# backup ke disk, biar tidak perlu re-embed kalau notebook restart
np.save("embeddings_5000.npy", embeddings)

Batches:   0%|          | 0/79 [00:00<?, ?it/s]

Batches: 100%|██████████| 79/79 [01:06<00:00,  1.19it/s]

(5000, 384)


## 4. Setup ChromaDB & Indexing

In [7]:
CHROMA_DIR = "chroma_db"
COLLECTION_NAME = "resep_masakan"

client = chromadb.PersistentClient(path=CHROMA_DIR)

# hapus collection lama kalau mau rebuild dari nol (WAJIB uncomment kalau kamu baru
# saja memperbaiki notebook 01 dan mau memastikan index tidak bercampur data lama)
# client.delete_collection(COLLECTION_NAME)

collection = client.get_or_create_collection(
    name=COLLECTION_NAME, metadata={"hnsw:space": "cosine"}
)
print(collection.count(), "dokumen sudah ada di collection sebelum insert")

0 dokumen sudah ada di collection sebelum insert


In [8]:
def build_metadata(row) -> dict:
    """ChromaDB metadata cuma boleh str/int/float/bool -> list/dict di-serialize jadi string.
    Catatan: setelah dibaca dari parquet, kolom list bisa berubah jadi numpy.ndarray,
    jadi di-convert ke list dulu sebelum json.dumps (kalau tidak, akan muncul error
    'TypeError: Object of type ndarray is not JSON serializable')."""
    ingredient_names = row["ingredient_names"]
    if isinstance(ingredient_names, np.ndarray):
        ingredient_names = ingredient_names.tolist()
    return {
        "title": row["title_clean"],
        "ingredient_names": json.dumps(list(ingredient_names), ensure_ascii=False),
        "num_ingredients": int(row["num_ingredients_clean"]),
        "steps_text": row["steps_text"][:2000],  # jaga2 kalau ada steps sangat panjang
    }


metadatas = [build_metadata(row) for _, row in df.iterrows()]
documents = df["text_for_embedding"].tolist()
ids = df["recipe_id"].tolist()

print(metadatas[0])

{'title': 'Martabak Mie', 'ingredient_names': '["mie instan", "wortel", "buncis", "telur ayam", "kornet kambing", "seledri", "daun bawang", "kulit lumpia"]', 'num_ingredients': 8, 'steps_text': '1. Rebus wortel hingga setengah matang, kemudian masukkan mie instan. Masak hingga mie setengah matang kemudian tambahkan buncis. Masak hingga semua matang. Matikan api, tiriskan 2. Masukkan daun bawang, seledri, kornet, bumbu mie instan dan telur. Aduk rata 3. Siapkan kulit lumpia, beri 1 sdm adonan mie, lipat seperti amplop. Panaskan wajan, beri minyak. Goreng martabak mie hingga kuning kecoklatan, cukup sekali balik. Angkat, tiriskan dan hidangkan.'}


In [9]:
# insert per batch (ChromaDB kadang error kalau insert sekaligus terlalu besar)
BATCH = 500

for start in range(0, len(df), BATCH):
    end = start + BATCH
    collection.upsert(
        ids=ids[start:end],
        embeddings=embeddings[start:end].tolist(),
        documents=documents[start:end],
        metadatas=metadatas[start:end],
    )
    print(f"Inserted {end if end < len(df) else len(df)} / {len(df)}")

print("Total dokumen di collection:", collection.count())

Inserted 500 / 5000
Inserted 1000 / 5000
Inserted 1500 / 5000
Inserted 2000 / 5000
Inserted 2500 / 5000
Inserted 3000 / 5000
Inserted 3500 / 5000
Inserted 4000 / 5000
Inserted 4500 / 5000
Inserted 5000 / 5000
Total dokumen di collection: 5000


## 5. Tes Retrieval (belum reranking efisiensi bahan)

Simulasi input user: `"2 butir telur, 100gr dada ayam, 2 siung bawang putih"`

In [10]:
def search_recipes(user_ingredients_text: str, top_k: int = 5):
    query_text = add_query_prefix(user_ingredients_text)
    query_embedding = model.encode(
        [query_text], normalize_embeddings=True, convert_to_numpy=True
    )

    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=top_k,
        include=["documents", "metadatas", "distances"],
    )
    return results


user_input = "2 butir telur, 100gr dada ayam, 2 siung bawang putih"
results = search_recipes(user_input, top_k=5)

for i in range(len(results["ids"][0])):
    rid = results["ids"][0][i]
    dist = results["distances"][0][i]
    meta = results["metadatas"][0][i]
    sbert_score = 1 - dist  # cosine distance -> cosine similarity
    print(f"[{i + 1}] {meta['title']}  (sbert_score={sbert_score:.4f})")
    print(f"    bahan: {meta['ingredient_names']}")
    print()

[1] Telur Dadar Daun Bawang  (sbert_score=0.7709)
    bahan: ["telur ayam", "daun bawang merah", "cabai", "garam", "merica bubuk"]

[2] Telur Dadar Tempe  (sbert_score=0.7222)
    bahan: ["telur ayam", "tempe", "daun bawang", "cabe rawit besar", "garam", "kaldu bubuk", "minyak goreng"]

[3] Telur Dadar  (sbert_score=0.7207)
    bahan: ["telur ayam", "kornet sapi", "bawang daun", "keju cheddar", "bawang putih bubuk", "lada bubuk", "garam"]

[4] Telur Dadar  (sbert_score=0.7188)
    bahan: ["telur", "bawang merah", "bawang putih", "daun bawang", "garam dan penyedap", "minyak goreng"]

[5] Telur Dadar Tempe  (sbert_score=0.7117)
    bahan: ["telur ayam", "tempe", "bawang putih", "cabe rawit merah", "daun jeruk", "minyak goreng", "garam dan kaldu"]



## 6. Diagnostic: Cek Kesegaran Index

Jalankan ini kapanpun kamu curiga index masih pakai data lama (misalnya setelah
memperbaiki parser di notebook 01, tapi lupa uncomment `delete_collection` di atas).

In [11]:
sample = collection.get(limit=5000, include=["metadatas"])

# cek kontaminasi satuan yang lolos parsing (harusnya 0 kalau index sudah fresh)
contaminated = 0
for m in sample["metadatas"]:
    names = json.loads(m["ingredient_names"])
    if any(n.startswith(("gr ", "btr ", "bh ", "ml ")) for n in names):
        contaminated += 1

print("Jumlah dokumen ter-index:", collection.count())
print(
    "Jumlah resep dgn nama bahan masih kecemar satuan (gr/btr/bh/ml di depan):",
    contaminated,
)
print(
    "-> Kalau angka ini TIDAK 0, index kamu STALE. Rebuild: uncomment delete_collection"
)
print("   di sub-bab 4, lalu jalankan ulang notebook ini dari load parquet.")

Jumlah dokumen ter-index: 5000
Jumlah resep dgn nama bahan masih kecemar satuan (gr/btr/bh/ml di depan): 0
-> Kalau angka ini TIDAK 0, index kamu STALE. Rebuild: uncomment delete_collection
   di sub-bab 4, lalu jalankan ulang notebook ini dari load parquet.
